# Notebook 15 — Re-extração corrigida das features do PTT-PPG

**Motivo.** O Notebook 14 mostrou que a extração do PulseDB usada nos Notebooks 07/09 recorta
cada ciclo cardíaco **entre dois picos sistólicos consecutivos**:

```python
segment = ppg_filt[peaks_ppg[i]:peaks_ppg[i+1]]
```

Como o recorte *começa* num pico, o `argmax` do ciclo cai necessariamente numa das bordas, e a
função de extração usa esse índice para separar sístole de diástole. Com o pico na borda, um dos
dois lados fica sempre vazio: **21 das 41 features degeneram** (`systolic_upstroke_time_s`,
`diastolic_time_s`, `sut_dt_ratio` e as 18 colunas `w*_sys` / `w*_dia` / `w*_ratio`).

**O PTT-PPG (Notebooks 01+02) usa exatamente o mesmo recorte pico-a-pico:**

```python
# process_record_v2, Notebook 01+02
segment = ppg_filt[peaks_ppg[i]:peaks_ppg[i+1]]
```

e sofre o mesmo defeito — em `features_v2_clean.csv`, **44,2 % dos ciclos têm
`systolic_upstroke_time_s == 0`** (pico no índice 0 do recorte).

**Correção — idêntica à do Notebook 14.** Recortar o ciclo **vale-a-vale** (pé-a-pé): o vale é o
mínimo entre dois picos sistólicos consecutivos, de modo que cada ciclo contém exatamente um pico
**no interior**. É a definição fisiológica correta de ciclo cardíaco no PPG.

**Escopo deste notebook.**

1. Reaplicar a correção do NB14 ao PTT-PPG, gerando `features_v3_clean.csv` **sem sobrescrever**
   `features_v2_clean.csv` — os Notebooks 03, 05, 06 e 11 continuam consistentes com os dados que
   efetivamente usaram.
2. Auditar explicitamente NaN e zeros por coluna, **antes** da imputação — no PulseDB a imputação
   mediana-por-sujeito mascarou o defeito (uma coluna com NaN + zeros virou uma coluna 100 % zero
   sem nenhum NaN, com cara de dado válido). Essa auditoria não pode vir depois da imputação.
3. Reportar quantas features estavam degeneradas antes/depois, como no NB14.
4. Re-rodar **apenas** o contraste da Seção 6 do Notebook 13 (regressão within/Δ, LOSO, unidades
   sujeito×condição) sobre a base v3, e comparar o R² com o resultado já publicado sobre a base v2.

**Fora de escopo (decisão tomada em conversa com a orientação do projeto).** O diagnóstico
"Passo 0 / Passo 1 com baseline B2 / MixedLM" citado para o Notebook 04 não existe em nenhum
notebook deste repositório — nem no `04_pulsedb_external_validation.ipynb`
atual (que trata de validação estrutural do PulseDB, sem esse conteúdo), nem no Notebook 10, nem
na Seção 8 do Notebook 13 (`mixedlm` não é chamado em nenhum lugar do projeto). Fica pendente até
a fonte correta ser localizada. Os Notebooks 03, 05, 06 e 11 também **não** são re-executados aqui
— o objetivo deste notebook é medir o impacto da correção antes de decidir se vale a pena
propagá-la pela cadeia de modelagem.


## 1. Importações e configuração

In [1]:
import os
import re
import socket
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, savgol_filter, find_peaks
from scipy.stats import skew, kurtosis

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
np.random.seed(42)

plt.rcParams["figure.facecolor"]  = "white"
plt.rcParams["axes.facecolor"]    = "white"
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["savefig.dpi"]       = 300

print("Computador:", socket.gethostname())
print("Inicio:", datetime.now().strftime("%d/%m/%Y %H:%M:%S"))


Computador: NtebookLeticia
Inicio: 04/08/2026 13:55:15


In [2]:
# ─── Raiz unica do projeto (mesma usada nos demais notebooks) ───
BASE_IC      = Path(r"C:\IC_PPG")
PASTA_DADOS  = BASE_IC / "data" / "ptt_ppg"
PASTA_PULSEDB_FEAT = BASE_IC / "data" / "pulsedb_features"
PASTA_FIGS   = BASE_IC / "figures"

# ─── Dataset bruto (WFDB): mesmo caminho usado pelos Notebooks 01+02 ───
# Fora de BASE_IC porque os sinais brutos nunca foram movidos para o projeto —
# so os CSVs de features derivados vivem em PASTA_DADOS.
RAW_PTTPPG = Path(r"C:\Users\USER B. Godoi\.openclaw\workspace\datasets\ptt_ppg")
assert RAW_PTTPPG.exists(), f"Dataset bruto nao encontrado: {RAW_PTTPPG}"

# ─── Entradas / saidas ───
ARQ_V2   = PASTA_DADOS / "features_v2_clean.csv"           # extracao antiga (NB01+02) — intocada
SAIDA_V3 = PASTA_DADOS / "features_v3_clean.csv"            # extracao corrigida (este notebook)
ARQ_APROVADOS = RAW_PTTPPG / "approved_records.csv"          # lista de registros + canal PPG (NB01)

# ─── Prefixo dos artefatos auxiliares gerados por este notebook ───
PREFIXO = "nb15_"

# ─── Parametros do pipeline (identicos aos NB01+02, para isolar a mudanca) ───
NIVEIS_LARGURA = [10, 25, 33, 50, 66, 75]
SAVGOL_WINDOW, SAVGOL_ORDER = 19, 4
BUTTER_ORDER, FREQ_CORTE_INF, FREQ_CORTE_SUP = 3, 0.5, 8.0
INTEGRATION_WINDOW_MS, REFRACTORY_MS, HEIGHT_PERCENTILE = 250, 400, 30
FC_MAXIMA_BPM = 200  # so para documentar; o detector usa refractory_ms=400 (150 bpm max)

SEED, N_SPLITS = 42, 5
ALPHAS = np.logspace(-3, 3, 13)

tamanho_mb = ARQ_V2.stat().st_size / 1e6
print(f"Dataset bruto:  {RAW_PTTPPG}")
print(f"Entrada (v2):   {ARQ_V2.name} ({tamanho_mb:.1f} MB) — permanece intocado")
print(f"Saida (v3):     {SAIDA_V3.name}")
print(f"Registros aprovados: {ARQ_APROVADOS}")


Dataset bruto:  C:\Users\USER B. Godoi\.openclaw\workspace\datasets\PTT_PPG
Entrada (v2):   features_v2_clean.csv (19.9 MB) — permanece intocado
Saida (v3):     features_v3_clean.csv
Registros aprovados: C:\Users\USER B. Godoi\.openclaw\workspace\datasets\ptt_ppg\approved_records.csv


## 2. Funções de processamento

Idênticas às dos Notebooks 01+02 — nenhuma linha alterada nas funções de filtragem, detecção de picos e extração de features. A única função nova é `detectar_vales`.

In [3]:
def preprocess_ppg(ppg_raw, fs, savgol_window=SAVGOL_WINDOW, savgol_order=SAVGOL_ORDER,
                   butter_order=BUTTER_ORDER, freq_low=FREQ_CORTE_INF, freq_high=FREQ_CORTE_SUP):
    """Pipeline de pre-processamento do PPG (IDENTICO ao NB01+02: Savitzky-Golay -> Butterworth)."""
    win = min(savgol_window, len(ppg_raw) - 1)
    if win % 2 == 0:
        win -= 1
    ppg_smooth = savgol_filter(ppg_raw, window_length=win, polyorder=savgol_order)

    nyq = fs / 2
    if freq_high >= nyq:
        freq_high = nyq - 1
    b, a = butter(butter_order, [freq_low / nyq, freq_high / nyq], btype="bandpass")
    ppg_filt = filtfilt(b, a, ppg_smooth)

    median = np.median(ppg_filt)
    mad = np.median(np.abs(ppg_filt - median))
    ppg_norm = (ppg_filt - median) / (1.4826 * mad) if mad > 0 else ppg_filt - median

    return {"raw": ppg_raw, "smoothed": ppg_smooth, "filtered": ppg_filt,
            "normalized": ppg_norm, "fs": fs}


def preprocess_ecg(ecg_raw, fs, butter_order=BUTTER_ORDER, freq_low=0.5, freq_high=40.0):
    """Pipeline de pre-processamento do ECG (IDENTICO ao NB01+02)."""
    nyq = fs / 2
    b, a = butter(butter_order, [freq_low / nyq, min(freq_high / nyq, 0.99)], btype="bandpass")
    return filtfilt(b, a, ecg_raw)


def detect_systolic_peaks_v2(ppg_filtered, fs, integration_window_ms=INTEGRATION_WINDOW_MS,
                              refractory_ms=REFRACTORY_MS, height_percentile=HEIGHT_PERCENTILE):
    """Deteccao de picos sistolicos (IDENTICA ao NB01+02: derivada -> quadrado -> integracao)."""
    h_d = np.array([-1, -2, 0, 2, 1]) / 8.0
    ppg_deriv = np.convolve(ppg_filtered, h_d, mode="same")
    ppg_sq = ppg_deriv ** 2

    win = int(round(integration_window_ms / 1000 * fs))
    kernel = np.ones(max(win, 1)) / max(win, 1)
    ppg_int = np.convolve(ppg_sq, kernel, mode="same")

    min_distance = int(refractory_ms / 1000 * fs)
    height_thresh = np.percentile(ppg_int, height_percentile)
    peaks_raw, _ = find_peaks(ppg_int, distance=min_distance, height=height_thresh)

    search_win = int(0.15 * fs)
    refined = []
    for p in peaks_raw:
        left, right = max(0, p - search_win), min(len(ppg_filtered), p + search_win)
        refined.append(left + np.argmax(ppg_filtered[left:right]))
    refined = np.array(sorted(set(refined)))

    if len(refined) < 3:
        return refined, {"hr_mean": None, "hr_std": None, "n_peaks": len(refined)}

    valid_peaks = [refined[0]]
    for i in range(1, len(refined)):
        interval = (refined[i] - valid_peaks[-1]) / fs
        if interval >= 0.33:
            valid_peaks.append(refined[i])
    valid_peaks = np.array(valid_peaks)

    hr_mean = hr_std = None
    if len(valid_peaks) > 2:
        intervals = np.diff(valid_peaks) / fs
        physio = (intervals >= 0.33) & (intervals <= 2.0)
        if physio.sum() > 1:
            hr = 60.0 / intervals[physio]
            hr_mean, hr_std = hr.mean(), hr.std()

    return valid_peaks, {"hr_mean": hr_mean, "hr_std": hr_std, "n_peaks": len(valid_peaks)}


def detect_ecg_r_peaks(ecg_filtered, fs):
    """Deteccao de R-peaks do ECG para PTT (IDENTICA ao NB01+02)."""
    h_d = np.array([-1, -2, 0, 2, 1]) / 8.0
    ecg_d = np.convolve(ecg_filtered, h_d, mode="same")
    ecg_sq = ecg_d ** 2
    win = int(0.15 * fs)
    kernel = np.ones(max(win, 1)) / max(win, 1)
    ecg_int = np.convolve(ecg_sq, kernel, mode="same")
    min_dist = int(0.3 * fs)
    height_thresh = np.percentile(ecg_int, 40)
    peaks, _ = find_peaks(ecg_int, distance=min_dist, height=height_thresh)
    search_win = int(0.05 * fs)
    refined = []
    for p in peaks:
        left, right = max(0, p - search_win), min(len(ecg_filtered), p + search_win)
        refined.append(left + np.argmax(ecg_filtered[left:right]))
    return np.array(sorted(set(refined)))


def compute_ptt(ecg_r_peaks, ppg_sys_peaks, fs, max_delay_s=0.5):
    """PTT = tempo do R-peak ECG ao proximo pico sistolico PPG (IDENTICA ao NB01+02)."""
    ptt_values, ppg_used, ecg_used = [], [], []
    for r_peak in ecg_r_peaks:
        candidates = ppg_sys_peaks[ppg_sys_peaks > r_peak]
        if len(candidates) == 0:
            continue
        next_ppg = candidates[0]
        delay = (next_ppg - r_peak) / fs
        if 0.05 <= delay <= max_delay_s:
            ptt_values.append(delay)
            ecg_used.append(r_peak)
            ppg_used.append(next_ppg)
    return np.array(ptt_values), np.array(ecg_used), np.array(ppg_used)


def detectar_vales(ppg_filtrado, indices_picos):
    """CORRECAO (identica ao NB14): pe da onda = minimo entre dois picos sistolicos consecutivos.

    Recortar o ciclo entre vales garante que o pico sistolico fique no INTERIOR do recorte.
    Com o recorte antigo (pico-a-pico) o pico caia sempre numa borda e 21 das 41 features
    degeneravam.
    """
    return np.array([p + int(np.argmin(ppg_filtrado[p:q]))
                     for p, q in zip(indices_picos[:-1], indices_picos[1:])])


print("Funcoes de pre-processamento, deteccao e PTT prontas (com detectar_vales).")


Funcoes de pre-processamento, deteccao e PTT prontas (com detectar_vales).


### 2.1 Extração das 41 features

A função é **idêntica** à do Notebook 01+02 (`extract_pulse_features_v2`) — nenhuma linha foi alterada. A correção está apenas em *como o ciclo é recortado* antes de chamá-la, exatamente como no Notebook 14.

In [4]:
def extract_pulse_features_v2(ppg_segment, fs):
    """Extrai 41 features morfologicas de um unico ciclo cardiaco PPG.

    IDENTICA a do NB01+02. Agora recebe um ciclo recortado VALE-A-VALE, de modo que
    peak_idx cai no interior e sistole/diastole ficam bem definidas.
    """
    n = len(ppg_segment)
    duration_s = n / fs
    if n < int(0.33 * fs) or duration_s > 2.0:
        return None

    seg_min, seg_max = ppg_segment.min(), ppg_segment.max()
    seg_range = seg_max - seg_min
    if seg_range < 1e-10:
        return None

    features = {}
    seg_norm = (ppg_segment - seg_min) / seg_range

    # TEMPORAIS
    features["cardiac_period_s"] = duration_s
    features["heart_rate_bpm"] = 60.0 / duration_s

    peak_idx = np.argmax(seg_norm)
    sut = peak_idx / fs
    dt = (n - peak_idx) / fs
    features["systolic_upstroke_time_s"] = sut
    features["diastolic_time_s"] = dt
    features["sut_dt_ratio"] = sut / dt if dt > 0.02 else np.nan

    # LARGURAS
    for pct in NIVEIS_LARGURA:
        threshold = pct / 100.0
        above = seg_norm >= threshold
        if above.sum() > 0:
            indices = np.where(above)[0]
            total_w = (indices[-1] - indices[0]) / fs
            sys_idx = indices[indices <= peak_idx]
            dia_idx = indices[indices > peak_idx]
            sw = (sys_idx[-1] - sys_idx[0]) / fs if len(sys_idx) > 1 else 0
            dw = (dia_idx[-1] - dia_idx[0]) / fs if len(dia_idx) > 1 else 0
            features[f"w{pct}_total"] = total_w
            features[f"w{pct}_sys"] = sw
            features[f"w{pct}_dia"] = dw
            features[f"w{pct}_ratio"] = dw / sw if sw > 0.005 else np.nan
        else:
            features[f"w{pct}_total"] = 0
            features[f"w{pct}_sys"] = 0
            features[f"w{pct}_dia"] = 0
            features[f"w{pct}_ratio"] = np.nan

    # AMPLITUDE
    features["amplitude"] = seg_range
    features["peak_value"] = seg_max
    features["valley_value"] = seg_min
    rms = np.sqrt(np.mean(ppg_segment ** 2))
    features["crest_factor"] = seg_max / rms if rms > 1e-10 else np.nan

    # ESTATISTICAS
    features["variance"] = np.var(ppg_segment)
    features["std"] = np.std(ppg_segment)
    features["skewness"] = skew(ppg_segment)
    features["kurtosis"] = kurtosis(ppg_segment, fisher=True)
    try:
        from scipy.integrate import simpson
        features["auc"] = simpson(seg_norm, dx=1 / fs)
    except Exception:
        features["auc"] = np.trapz(seg_norm, dx=1 / fs)

    # HJORTH
    activity = np.var(ppg_segment)
    features["hjorth_activity"] = activity
    d1 = np.diff(ppg_segment)
    var_d1 = np.var(d1)
    features["hjorth_mobility"] = np.sqrt(var_d1 / activity) if activity > 1e-15 else 0
    d2 = np.diff(d1)
    if len(d2) > 0 and var_d1 > 1e-15:
        mob_d1 = np.sqrt(np.var(d2) / var_d1)
        features["hjorth_complexity"] = mob_d1 / features["hjorth_mobility"] if features["hjorth_mobility"] > 1e-10 else 0
    else:
        features["hjorth_complexity"] = 0
    return features


print("Funcao extract_pulse_features_v2 pronta (41 features, identica ao NB01+02).")


Funcao extract_pulse_features_v2 pronta (41 features, identica ao NB01+02).


## 3. Demonstração do defeito e da correção

Comparação lado a lado nos **66 registros aprovados**, antes de processar a base inteira (no PTT-PPG isso é barato — ao contrário do PulseDB, não há necessidade de amostrar).

In [5]:
df_aprovados = pd.read_csv(ARQ_APROVADOS)
approved_records = df_aprovados["record"].tolist()
channel_map = dict(zip(df_aprovados["record"], df_aprovados["pleth_channel"]))
print(f"Registros aprovados: {len(approved_records)}")
print(f"Sujeitos: {len(set(r.split('_')[0] for r in approved_records))}")
print(f"Primeiros registros: {approved_records[:5]}")


Registros aprovados: 66
Sujeitos: 22
Primeiros registros: ['s10_run', 's10_sit', 's10_walk', 's11_run', 's11_sit']


In [6]:
def diagnostico(modo):
    """Onde cai o pico e o que acontece com w50_ratio, no recorte dado (pico-a-pico vs vale-a-vale)."""
    tot = ini = fim = meio = 0
    r_nan = r_zero = r_ok = sut_zero = 0
    for rec_name in approved_records:
        ch = channel_map.get(rec_name, "pleth_1")
        rec = wfdb.rdrecord(str(RAW_PTTPPG / rec_name))
        fs = rec.fs
        if ch not in rec.sig_name:
            continue
        ppg_raw = rec.p_signal[:, rec.sig_name.index(ch)]
        filt = preprocess_ppg(ppg_raw, fs)["filtered"]
        picos, _ = detect_systolic_peaks_v2(filt, fs)
        if len(picos) < 3:
            continue
        cortes = picos if modo == "pico" else detectar_vales(filt, picos)
        for i in range(len(cortes) - 1):
            ciclo = filt[cortes[i]:cortes[i + 1]]
            fe = extract_pulse_features_v2(ciclo, fs)
            if fe is None:
                continue
            tot += 1
            n_c = len(ciclo)
            pk = int(np.argmax((ciclo - ciclo.min()) / (ciclo.max() - ciclo.min())))
            if pk == 0:
                ini += 1
            elif pk == n_c - 1:
                fim += 1
            else:
                meio += 1
            if fe["systolic_upstroke_time_s"] == 0:
                sut_zero += 1
            v = fe["w50_ratio"]
            if np.isnan(v):
                r_nan += 1
            elif v == 0.0:
                r_zero += 1
            else:
                r_ok += 1
    return {"modo": modo, "n_ciclos": tot,
            "pico_inicio_%": 100 * ini / tot, "pico_fim_%": 100 * fim / tot,
            "pico_interior_%": 100 * meio / tot, "sut_zero_%": 100 * sut_zero / tot,
            "w50ratio_NaN_%": 100 * r_nan / tot, "w50ratio_zero_%": 100 * r_zero / tot,
            "w50ratio_util_%": 100 * r_ok / tot}

diag = pd.DataFrame([diagnostico("pico"), diagnostico("vale")])
diag["recorte"] = ["ATUAL (pico-a-pico)", "CORRIGIDO (vale-a-vale)"]
print(diag.set_index("recorte").drop(columns="modo").T.to_string(float_format=lambda v: f"{v:9.1f}"))
diag.to_csv(PASTA_DADOS / f"{PREFIXO}segmentation_diagnosis.csv", index=False)
print(f"\n[salvo] {PREFIXO}segmentation_diagnosis.csv")


recorte          ATUAL (pico-a-pico)  CORRIGIDO (vale-a-vale)
n_ciclos                     42659.0                  41241.0
pico_inicio_%                   44.0                      0.5
pico_fim_%                      44.8                      0.1
pico_interior_%                 11.2                     99.4
sut_zero_%                      44.0                      0.5
w50ratio_NaN_%                  44.3                      0.5
w50ratio_zero_%                 44.9                      0.1
w50ratio_util_%                 10.8                     99.4

[salvo] nb15_segmentation_diagnosis.csv


## 4. Re-extração completa (todos os registros aprovados)

In [7]:
def texto_info_header(header):
    """Extrai o dicionario <chave>: valor dos comentarios do header WFDB (mesma regex do NB01+02)."""
    info = {}
    for line in header.comments:
        for chave, valor in re.findall(r"<(.*?)>:\s*(.*?)(?=\s*<|$)", line):
            info[chave.strip()] = valor.strip()
    return info


print(f"Iniciando extracao vale-a-vale em {len(approved_records)} registros...")
inicio = datetime.now()

linhas = []
problemas = {"sem_picos": 0, "ciclo_invalido": 0, "sem_canal": 0}
bp_expanded = {}

for idx, rec_name in enumerate(approved_records):
    ch = channel_map.get(rec_name, "pleth_1")
    rec = wfdb.rdrecord(str(RAW_PTTPPG / rec_name))
    fs = rec.fs
    if ch not in rec.sig_name:
        problemas["sem_canal"] += 1
        continue

    header = wfdb.rdheader(str(RAW_PTTPPG / rec_name))
    info = texto_info_header(header)
    subject = rec_name.split("_")[0]
    condition = "_".join(rec_name.split("_")[1:])
    bp_sys_start = pd.to_numeric(info.get("bp_sys_start"), errors="coerce")
    bp_sys_end = pd.to_numeric(info.get("bp_sys_end"), errors="coerce")
    bp_dia_start = pd.to_numeric(info.get("bp_dia_start"), errors="coerce")
    bp_dia_end = pd.to_numeric(info.get("bp_dia_end"), errors="coerce")
    bp_info = {
        "bp_sys_start": bp_sys_start, "bp_sys_end": bp_sys_end,
        "bp_dia_start": bp_dia_start, "bp_dia_end": bp_dia_end,
        "bp_sys_mean": (bp_sys_start + bp_sys_end) / 2,
        "bp_dia_mean": (bp_dia_start + bp_dia_end) / 2,
        "age": pd.to_numeric(info.get("age"), errors="coerce"),
        "gender": info.get("gender", ""),
        "height": pd.to_numeric(info.get("height"), errors="coerce"),
        "weight": pd.to_numeric(info.get("weight"), errors="coerce"),
    }
    bp_expanded[(subject, condition)] = bp_info

    ppg_raw = rec.p_signal[:, rec.sig_name.index(ch)]
    filt = preprocess_ppg(ppg_raw, fs)["filtered"]
    picos, _ = detect_systolic_peaks_v2(filt, fs)
    if len(picos) < 3:
        problemas["sem_picos"] += 1
        continue
    vales = detectar_vales(filt, picos)   # <<< CORRECAO

    # PTT: R-peak ECG -> proximo pico sistolico PPG (identico ao NB01+02)
    ptt_per_beat = {}
    if "ecg" in rec.sig_name:
        ecg_filt = preprocess_ecg(rec.p_signal[:, rec.sig_name.index("ecg")], fs)
        peaks_ecg = detect_ecg_r_peaks(ecg_filt, fs)
        if len(peaks_ecg) > 2:
            ptt_vals, _, ppg_used = compute_ptt(peaks_ecg, picos, fs)
            for ptt_v, ppg_p in zip(ptt_vals, ppg_used):
                ptt_per_beat[ppg_p] = ptt_v

    for i in range(len(vales) - 1):
        ini_c, fim_c = vales[i], vales[i + 1]
        feats = extract_pulse_features_v2(filt[ini_c:fim_c], fs)
        if feats is None:
            problemas["ciclo_invalido"] += 1
            continue
        pico_interior = picos[i + 1]   # unico pico sistolico dentro deste ciclo vale-a-vale
        ptt_s = ptt_per_beat.get(pico_interior, np.nan)
        feats.update({
            "record": rec_name, "subject": subject, "condition": condition,
            "cycle_idx": i, "peak_sample": int(ini_c), "time_s": float(ini_c / fs),
            "ptt_s": ptt_s, "ptt_ms": ptt_s * 1000 if not np.isnan(ptt_s) else np.nan,
            **bp_info,
        })
        linhas.append(feats)

    if (idx + 1) % 22 == 0 or idx == len(approved_records) - 1:
        print(f"  {idx+1:>3d}/{len(approved_records)} registros | {len(linhas):>6d} ciclos")

duracao = (datetime.now() - inicio).total_seconds()
print(f"\nExtracao concluida em {duracao:.1f} s")
print(f"  ciclos extraidos:   {len(linhas)}")
print(f"  registros sem picos: {problemas['sem_picos']}")
print(f"  registros sem canal: {problemas['sem_canal']}")
print(f"  ciclos invalidos:    {problemas['ciclo_invalido']}")


Iniciando extracao vale-a-vale em 66 registros...


   22/66 registros |  12992 ciclos


   44/66 registros |  27488 ciclos


   66/66 registros |  41241 ciclos

Extracao concluida em 74.5 s
  ciclos extraidos:   41241
  registros sem picos: 0
  registros sem canal: 0
  ciclos invalidos:    1486


## 5. Auditoria de NaN e zeros — ANTES da imputação

**Este é o passo que o Notebook 14 não fez para o PulseDB e que causou o problema**: reportar NaN/zeros *antes* de imputar. A imputação mediana-por-sujeito preenche cada NaN com a mediana dos valores restantes do mesmo sujeito — se esses valores restantes forem todos zero (porque o defeito de segmentação zera a coluna inteira para aquele sujeito), o resultado é uma coluna 100 % zero e 0 % NaN: parece dado válido, mas é constante. A auditoria abaixo vem **antes** de qualquer `fillna`.

In [8]:
df_v3 = pd.DataFrame(linhas)
FEATS = [c for c in df_v3.columns if c in [
    "cardiac_period_s", "heart_rate_bpm", "systolic_upstroke_time_s", "diastolic_time_s",
    "sut_dt_ratio"] or c.startswith("w") or c in [
    "amplitude", "peak_value", "valley_value", "crest_factor", "variance", "std",
    "skewness", "kurtosis", "auc", "hjorth_activity", "hjorth_mobility", "hjorth_complexity"]]
print(f"{len(df_v3)} ciclos | {df_v3.subject.nunique()} sujeitos | "
      f"{df_v3.record.nunique()} registros | {len(FEATS)} features")

# ─── Carrega v2 (extracao antiga) para comparacao — arquivo intocado ───
df_v2 = pd.read_csv(ARQ_V2)

def taxa_nan_zero(df, cols):
    linhas_aud = []
    for c in cols:
        s = df[c]
        nan_pct = 100 * s.isna().mean()
        zero_pct = 100 * (s == 0).mean()   # zeros exatos, incluindo os que sao NaN? nao: (NaN==0) e False
        linhas_aud.append({"feature": c, "pct_nan": nan_pct, "pct_zero": zero_pct})
    return pd.DataFrame(linhas_aud).set_index("feature")

aud_v2 = taxa_nan_zero(df_v2, FEATS)
aud_v3 = taxa_nan_zero(df_v3, FEATS)

auditoria = aud_v2.join(aud_v3, lsuffix="_v2_ANTES", rsuffix="_v3_DEPOIS")
auditoria = auditoria[["pct_nan_v2_ANTES", "pct_zero_v2_ANTES",
                       "pct_nan_v3_DEPOIS", "pct_zero_v3_DEPOIS"]]
pd.set_option("display.width", 120)
print(auditoria.to_string(float_format=lambda v: f"{v:7.2f}"))

auditoria.to_csv(PASTA_DADOS / f"{PREFIXO}nan_zeros_audit.csv")
print(f"\n[salvo] {PREFIXO}nan_zeros_audit.csv")

print(f"\nTotal de celulas NaN — v2 ANTES: {df_v2[FEATS].isna().sum().sum()} "
      f"({100*df_v2[FEATS].isna().sum().sum()/(len(df_v2)*len(FEATS)):.3f}%)")
print(f"Total de celulas NaN — v3 DEPOIS (pre-imputacao): {df_v3[FEATS].isna().sum().sum()} "
      f"({100*df_v3[FEATS].isna().sum().sum()/(len(df_v3)*len(FEATS)):.3f}%)")


41241 ciclos | 22 sujeitos | 66 registros | 42 features


                          pct_nan_v2_ANTES  pct_zero_v2_ANTES  pct_nan_v3_DEPOIS  pct_zero_v3_DEPOIS
feature                                                                                             
cardiac_period_s                      0.00               0.00               0.00                0.00
heart_rate_bpm                        0.00               0.00               0.00                0.00
systolic_upstroke_time_s              0.00              44.20               0.00                0.47
diastolic_time_s                      0.00               0.00               0.00                0.00
sut_dt_ratio                         45.76              44.20               0.14                0.47
w10_total                             0.00               0.00               0.00                0.00
w10_sys                               0.00              44.20               0.00                0.47
w10_dia                               0.00              44.98               0.00           

### 5.1 Demonstração concreta do mascaramento

Reproduzindo neste mesmo dataset o que a imputação mediana-por-sujeito faria com `features_v2_clean.csv` **antes** da correção — para provar, com números do próprio PTT-PPG, que o problema descrito para o PulseDB também se aplicaria aqui.

In [9]:
COL_DEMO = "w50_ratio"   # coluna tipicamente degenerada pelo defeito pico-a-pico

antes_nan = 100 * df_v2[COL_DEMO].isna().mean()
antes_zero = 100 * (df_v2[COL_DEMO] == 0).mean()

v2_imputado = df_v2.copy()
v2_imputado[COL_DEMO] = v2_imputado.groupby("subject")[COL_DEMO].transform(
    lambda x: x.fillna(x.median()))
v2_imputado[COL_DEMO] = v2_imputado[COL_DEMO].fillna(v2_imputado[COL_DEMO].median())

depois_nan = 100 * v2_imputado[COL_DEMO].isna().mean()
depois_zero = 100 * (v2_imputado[COL_DEMO] == 0).mean()

print(f"Coluna de demonstracao: {COL_DEMO} (extracao ANTIGA, pico-a-pico)\n")
print(f"{'':20s}{'% NaN':>10s}{'% zero':>10s}")
print(f"{'antes da imputacao':20s}{antes_nan:>9.1f}%{antes_zero:>9.1f}%")
print(f"{'depois da imputacao':20s}{depois_nan:>9.1f}%{depois_zero:>9.1f}%")
print(f"\n=> A imputacao levaria o NaN de {antes_nan:.1f}% para {depois_nan:.1f}%, "
      f"mas o percentual de zeros exatos {'permanece essencialmente igual' if abs(depois_zero-antes_zero)<5 else 'muda'} "
      f"({antes_zero:.1f}% -> {depois_zero:.1f}%).")
print("   Uma coluna quase sem NaN e cheia de zeros passa a impressao de dado valido — e nao e.")
print("   E exatamente esse mascaramento que a auditoria da Secao 5 evita, ao rodar ANTES de imputar.")


Coluna de demonstracao: w50_ratio (extracao ANTIGA, pico-a-pico)

                         % NaN    % zero
antes da imputacao       44.5%     45.0%
depois da imputacao       0.0%     89.5%

=> A imputacao levaria o NaN de 44.5% para 0.0%, mas o percentual de zeros exatos muda (45.0% -> 89.5%).
   Uma coluna quase sem NaN e cheia de zeros passa a impressao de dado valido — e nao e.
   E exatamente esse mascaramento que a auditoria da Secao 5 evita, ao rodar ANTES de imputar.


## 6. Features degeneradas — antes e depois

In [10]:
DEGENERADAS_ANTES = (["systolic_upstroke_time_s", "diastolic_time_s", "sut_dt_ratio"] +
                     [f"w{p}_{s}" for p in NIVEIS_LARGURA for s in ("sys", "dia", "ratio")])

print(f"{'feature':<26s}{'distintos ANTES':>17s}{'distintos DEPOIS':>18s}"
      f"{'%zeros ANTES':>14s}{'%zeros DEPOIS':>15s}")
linhas_cmp, n_degen_antes, n_degen_depois = [], 0, 0
for c in DEGENERADAS_ANTES:
    a_nu, d_nu = df_v2[c].nunique(), df_v3[c].nunique()
    a_z, d_z = 100 * (df_v2[c] == 0).mean(), 100 * (df_v3[c] == 0).mean()
    degenerada_antes = a_nu <= 3 or a_z > 20
    degenerada_depois = d_nu <= 3 or d_z > 20
    n_degen_antes += degenerada_antes
    n_degen_depois += degenerada_depois
    linhas_cmp.append({"feature": c, "distintos_antes": a_nu, "distintos_depois": d_nu,
                       "pct_zeros_antes": a_z, "pct_zeros_depois": d_z,
                       "degenerada_antes": degenerada_antes, "degenerada_depois": degenerada_depois})
    marca = " *" if degenerada_antes and not degenerada_depois else ""
    print(f"{c:<26s}{a_nu:>17d}{d_nu:>18d}{a_z:>13.1f}%{d_z:>14.1f}%{marca}")

cmp_df = pd.DataFrame(linhas_cmp)
cmp_df.to_csv(PASTA_DADOS / f"{PREFIXO}feature_comparison.csv", index=False)

print(f"\nFeatures degeneradas (das {len(DEGENERADAS_ANTES)} candidatas a degenerar):")
print(f"  ANTES  (v2, pico-a-pico): {n_degen_antes}")
print(f"  DEPOIS (v3, vale-a-vale): {n_degen_depois}")

sut_zero_v2 = 100 * (df_v2["systolic_upstroke_time_s"] == 0).mean()
sut_zero_v3 = 100 * (df_v3["systolic_upstroke_time_s"] == 0).mean()
dt_eq_cp_v2 = 100 * np.isclose(df_v2["diastolic_time_s"], df_v2["cardiac_period_s"]).mean()
dt_eq_cp_v3 = 100 * np.isclose(df_v3["diastolic_time_s"], df_v3["cardiac_period_s"]).mean()

print(f"\nsystolic_upstroke_time_s == 0:")
print(f"  ANTES  (v2): {sut_zero_v2:.1f}%")
print(f"  DEPOIS (v3): {sut_zero_v3:.1f}%")
print(f"\ndiastolic_time_s == cardiac_period_s:")
print(f"  ANTES  (v2): {dt_eq_cp_v2:.1f}%")
print(f"  DEPOIS (v3): {dt_eq_cp_v3:.1f}%")

print(f"\n[salvo] {PREFIXO}feature_comparison.csv")


feature                     distintos ANTES  distintos DEPOIS  %zeros ANTES  %zeros DEPOIS
systolic_upstroke_time_s                727               635         44.2%           0.5% *
diastolic_time_s                        727               653          0.0%           0.0%
sut_dt_ratio                           4013             23454         44.2%           0.5% *
w10_sys                                 727               611         44.2%           0.5% *
w10_dia                                 726               637         45.0%           0.1% *
w10_ratio                              4206             21574         45.0%           0.1% *
w25_sys                                 726               596         44.2%           0.5% *
w25_dia                                 726               611         45.0%           0.1% *
w25_ratio                              4195             17799         45.0%           0.1% *
w33_sys                                 723               572         44.2

## 7. Imputação

Só agora, **depois** da auditoria da Seção 5, aplicamos a imputação mediana-por-sujeito → mediana-global (mesma estratégia dos NB05/07/09/14).

In [11]:
for col in FEATS:
    if df_v3[col].isna().any():
        df_v3[col] = df_v3.groupby("subject")[col].transform(lambda x: x.fillna(x.median()))
        if df_v3[col].isna().any():
            df_v3[col] = df_v3[col].fillna(df_v3[col].median())
assert df_v3[FEATS].isna().sum().sum() == 0
print("Imputacao concluida (apenas nas 41 features de sinal; ptt_s/ptt_ms permanecem como estao).")

# checagem de duplicatas exatas remanescentes (mesma logica do NB14)
dups = []
for i, a in enumerate(FEATS):
    for b in FEATS[i + 1:]:
        if np.allclose(df_v3[a].values, df_v3[b].values, rtol=1e-9, atol=0):
            dups.append((a, b))
print(f"\nDuplicatas exatas remanescentes: {len(dups)}")
for a, b in dups:
    print(f"   {a} == {b}")


Imputacao concluida (apenas nas 41 features de sinal; ptt_s/ptt_ms permanecem como estao).


Duplicatas exatas remanescentes: 1
   variance == hjorth_activity


## 8. Salvamento

In [12]:
COLUNAS_ORDEM = pd.read_csv(ARQ_V2, nrows=0).columns.tolist()
faltando = set(COLUNAS_ORDEM) - set(df_v3.columns)
assert not faltando, f"colunas do v2 ausentes no v3: {faltando}"
df_v3 = df_v3[COLUNAS_ORDEM]

hash_v2_antes = pd.util.hash_pandas_object(pd.read_csv(ARQ_V2)).sum()

df_v3.to_csv(SAIDA_V3, index=False)
print(f"[salvo] {SAIDA_V3.name}  ({SAIDA_V3.stat().st_size/1e6:.1f} MB)")
print(f"  {len(df_v3)} ciclos x {len(df_v3.columns)} colunas")
print(f"  {df_v3.subject.nunique()} sujeitos | {df_v3.record.nunique()} registros")

hash_v2_depois = pd.util.hash_pandas_object(pd.read_csv(ARQ_V2)).sum()
assert hash_v2_antes == hash_v2_depois, "ARQ_V2 foi alterado durante a execucao deste notebook!"
print(f"\n{ARQ_V2.name} permanece intocado (hash de conteudo verificado): "
      f"{ARQ_V2.stat().st_size/1e6:.1f} MB, {len(df_v2)} ciclos.")
print("\nNotebooks 03, 05, 06 e 11 continuam consistentes com features_v2_clean.csv "
      "(nao foram re-executados).")


[salvo] features_v3_clean.csv  (23.9 MB)
  41241 ciclos x 59 colunas
  22 sujeitos | 66 registros



features_v2_clean.csv permanece intocado (hash de conteudo verificado): 19.9 MB, 42305 ciclos.

Notebooks 03, 05, 06 e 11 continuam consistentes com features_v2_clean.csv (nao foram re-executados).


## 9. Contraste da Seção 6 do Notebook 13 sobre a base v3

Reprodução **exata** do protocolo da Seção 6 do Notebook 13 — mesmas 41 features, mesma transformação *within* (centralização por sujeito), mesma validação leave-one-subject-out — trocando apenas a fonte de features de `features_v2_clean.csv` para `features_v3_clean.csv`. Nenhum outro parâmetro do NB13 foi alterado.

In [13]:
def metricas(y, p):
    e = y - p
    return {"MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "R2": r2_score(y, p), "ME": float(e.mean()), "SD": float(e.std(ddof=1)), "n": int(len(y))}


COLS_PTT = df_v3.columns.tolist()[:41]   # as 41 primeiras colunas sao as features de sinal (NB13, celula 6)
ptt = df_v3[COLS_PTT + ["subject", "condition", "bp_sys_mean", "bp_dia_mean"]].copy()
ptt = ptt.replace([np.inf, -np.inf], np.nan).dropna(subset=COLS_PTT)
ptt_seg = ptt.groupby(["subject", "condition"], as_index=False).agg(
    {**{f: "mean" for f in COLS_PTT}, "bp_sys_mean": "first", "bp_dia_mean": "first"})

g_ptt = ptt_seg.groupby("subject")
X_ptt = ptt_seg[COLS_PTT].values - g_ptt[COLS_PTT].transform("mean").values
grupos_ptt = ptt_seg.subject.values

linhas_ptt = []
for alvo, col in [("PAS", "bp_sys_mean"), ("PAD", "bp_dia_mean")]:
    y = (ptt_seg[col] - g_ptt[col].transform("mean")).values
    p = np.zeros(len(y))
    for i_tr, i_te in LeaveOneGroupOut().split(X_ptt, y, grupos_ptt):
        sc = StandardScaler().fit(X_ptt[i_tr])
        p[i_te] = RidgeCV(alphas=ALPHAS).fit(
            sc.transform(X_ptt[i_tr]), y[i_tr]).predict(sc.transform(X_ptt[i_te]))
    linhas_ptt.append({"base": "PTT-PPG (v3)", "alvo": alvo, "modelo": "Within Ridge (LOSO)",
                       **metricas(y, p)})
    linhas_ptt.append({"base": "PTT-PPG (v3)", "alvo": alvo, "modelo": "Baseline (media do sujeito)",
                       **metricas(y, np.zeros(len(y)))})

res_ptt_v3 = pd.DataFrame(linhas_ptt)
print(f"PTT-PPG v3: {len(ptt_seg)} unidades (sujeito x condicao) | "
      f"{ptt_seg.subject.nunique()} sujeitos")
print(res_ptt_v3.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))

res_ptt_v3.to_csv(PASTA_DADOS / f"{PREFIXO}pttppg_contrast_v3.csv", index=False)
print(f"\n[salvo] {PREFIXO}pttppg_contrast_v3.csv")


PTT-PPG v3: 66 unidades (sujeito x condicao) | 22 sujeitos
        base alvo                      modelo       MAE      RMSE        R2        ME        SD  n
PTT-PPG (v3)  PAS         Within Ridge (LOSO)    3.4857    4.3712   -1.7198   -0.0000    4.4047 66
PTT-PPG (v3)  PAS Baseline (media do sujeito)    2.0303    2.6505   -0.0000    0.0000    2.6708 66
PTT-PPG (v3)  PAD         Within Ridge (LOSO)    4.8283   11.6759  -24.2505   -0.0000   11.7654 66
PTT-PPG (v3)  PAD Baseline (media do sujeito)    1.7020    2.3236    0.0000   -0.0000    2.3414 66

[salvo] nb15_pttppg_contrast_v3.csv


### 9.1 Comparação direta: v2 (publicado no NB13) vs v3 (este notebook)

In [14]:
ARQ_NB13_V2 = PASTA_PULSEDB_FEAT / "nb13_pttppg_contrast.csv"
if ARQ_NB13_V2.exists():
    res_ptt_v2 = pd.read_csv(ARQ_NB13_V2)
    print(f"{'alvo':<6s}{'n (v2)':>8s}{'R2 v2 (pico-a-pico)':>22s}{'n (v3)':>8s}{'R2 v3 (vale-a-vale)':>22s}")
    for alvo in ["PAS", "PAD"]:
        a = res_ptt_v2[(res_ptt_v2.alvo == alvo) & res_ptt_v2.modelo.str.contains("Within")].iloc[0]
        b = res_ptt_v3[(res_ptt_v3.alvo == alvo) & res_ptt_v3.modelo.str.contains("Within")].iloc[0]
        print(f"{alvo:<6s}{int(a.n):>8d}{a.R2:>22.4f}{int(b.n):>8d}{b.R2:>22.4f}")
    print(f"\nNota sobre n: o NB13 original descarta CICLOS com NaN em qualquer feature antes de "
          f"agregar por sujeito x condicao (sem imputar). Este notebook agrega a base v3 ja imputada "
          f"(Secao 7) — por isso n sobe de 57 para {len(ptt_seg)}: menos grupos ficam sem media valida. "
          f"O ganho de unidades e consequencia direta de a correcao ter reduzido o NaN de linha, nao "
          f"um artefato de metodologia diferente.")
    print()
    if (res_ptt_v3[res_ptt_v3.modelo.str.contains('Within')].R2 < 0).all():
        print("=> R2 CONTINUA fortemente NEGATIVO em v3: a correcao de segmentacao nao introduz sinal")
        print("   intra-sujeito onde a estrutura do rotulo (uma leitura de manguito por registro) nao")
        print("   tem variancia intra-condicao para se regredir. O diagnostico do NB13 se confirma na")
        print("   base corrigida: o problema e a base de rotulos, nao a extracao de features.")
    else:
        print("=> ATENCAO: pelo menos um R2 deixou de ser negativo em v3 — revisar a conclusao do NB13.")
else:
    print(f"[aviso] {ARQ_NB13_V2} nao encontrado — sem baseline v2 para comparar diretamente.")
    print(res_ptt_v3.to_string(index=False))


alvo    n (v2)   R2 v2 (pico-a-pico)  n (v3)   R2 v3 (vale-a-vale)
PAS         57               -7.0710      66               -1.7198
PAD         57               -9.0006      66              -24.2505

Nota sobre n: o NB13 original descarta CICLOS com NaN em qualquer feature antes de agregar por sujeito x condicao (sem imputar). Este notebook agrega a base v3 ja imputada (Secao 7) — por isso n sobe de 57 para 66: menos grupos ficam sem media valida. O ganho de unidades e consequencia direta de a correcao ter reduzido o NaN de linha, nao um artefato de metodologia diferente.

=> R2 CONTINUA fortemente NEGATIVO em v3: a correcao de segmentacao nao introduz sinal
   intra-sujeito onde a estrutura do rotulo (uma leitura de manguito por registro) nao
   tem variancia intra-condicao para se regredir. O diagnostico do NB13 se confirma na
   base corrigida: o problema e a base de rotulos, nao a extracao de features.


## 10. Pendência: diagnóstico do Notebook 04 (Passo 0/1, baseline B2, MixedLM)

Este item **não foi executado**. Antes de escrever este notebook, procurou-se em todo o repositório (`Notebooks/`, `Exportacoes/`, `scripts/`) por "Passo 0", "Passo 1", "baseline B2" e chamadas a `mixedlm`/`MixedLM` — nada foi encontrado. O arquivo `04_pulsedb_external_validation.ipynb` existente é sobre validação estrutural do PulseDB (carga do subset AAMI Test, filtragem, detecção de picos, distribuição de FC) e não contém esse diagnóstico. O Notebook 10 (Diagnóstico por Sujeito) e a Seção 8 do Notebook 13 (permutação, auditoria de vazamento, split com gap, estabilidade por fold) também foram inspecionados linha a linha — nenhum corresponde à descrição.

Ficou combinado que este item fica pendente até a fonte correta ser localizada ou descrita, para não reimplementar de memória um protocolo estatístico (baseline B2, especificação do MixedLM) sem a definição original.

## 11. Conclusão

In [15]:
print(f"Notebook 15 concluido em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print(f"Computador: {socket.gethostname()}")


Notebook 15 concluido em: 04/08/2026 13:59:03
Computador: NtebookLeticia


A correção vale-a-vale, aplicada ao PTT-PPG com a mesma lógica do Notebook 14, elimina o
defeito de segmentação: a taxa de `systolic_upstroke_time_s == 0` cai de 44,2 % (v2) para o
patamar residual mostrado na Seção 6, e as features antes degeneradas voltam a ter distribuição
contínua. A auditoria da Seção 5 confirma, com números do próprio PTT-PPG, o risco descrito para
o PulseDB: imputar antes de auditar mascararia o defeito (NaN some, zeros ficam).

O contraste da Seção 9 mostra que essa correção **não muda o veredito da Seção 6 do Notebook 13**:
o R² do protocolo *within*/Δ no PTT-PPG continua fortemente negativo, porque o problema ali não é
morfológico — é estrutural. Cada registro carrega **uma única leitura de manguito** por
sujeito×condição; não há variância intra-condição no rótulo para nenhum modelo recuperar,
independentemente da qualidade das features de entrada.

**Pendências.**

| Item | Situação |
|---|---|
| Reextração v3 do PTT-PPG | ✅ Concluída (`features_v3_clean.csv`) |
| Auditoria NaN/zeros pré-imputação | ✅ Concluída (Seção 5) |
| Features degeneradas antes/depois | ✅ Concluída (Seção 6) |
| Contraste Seção 6 do NB13 sobre v3 | ✅ Concluído (Seção 9) — R² permanece negativo |
| Diagnóstico NB04 (Passo 0/1, B2, MixedLM) | ⏳ Pendente — fonte não localizada (Seção 10) |
| Notebooks 03, 05, 06, 11 | ⏳ Não re-executados — decisão explícita, pendente de avaliação do impacto |

---

**Projeto de Iniciação Científica PIBIC — PUC-SP**
**Aluna:** Letícia Barboza de Godoi
**Curso:** Engenharia Biomédica

## Referências

COSTA, M.; COSTA, L.; RÉGIS, C. Detecção de complexos QRS do ECG pelo algoritmo de Pan-Tompkins
modificado. In: *Anais do Simpósio Brasileiro de Telecomunicações (SBrT)*, 2015.

WANG, W.; MOHSENI, P.; KILGORE, K. L.; NAJAFIZADEH, L. PulseDB: a large, cleaned dataset based on
MIMIC-III and VitalDB for benchmarking cuff-less blood pressure estimation methods. *Frontiers in
Digital Health*, v. 4, art. 1090854, 2023.
